# Chapter 19 — Microstructural Features

Nine features spanning the three "generations" of market microstructure theory
AFML Ch19 walks through, scoped to what's derivable from a raw trade tape +
dollar bars (no order book, no cancellations, no options data).

Built out of chapter-sequence order, as a deliberate feature-enrichment effort:
three earlier chapters (Ch08/09's degenerate feature importance and weak CV
scores, Ch12's near-zero CPCV Sharpe distribution, and Ch13's non-stationary
O-U calibration) all independently pointed at the same root cause — the real
training table has essentially one feature (`fracdiff`). This notebook exists
to give it more.

**Path convention:** edit `AFML_ROOT` below for your machine (Jupyter's
working directory isn't reliably this notebook's own folder).

In [ ]:
AFML_ROOT = r'C:\ws\AFML'  # <-- EDIT THIS for your machine

import os, sys
if AFML_ROOT not in sys.path:
    sys.path.insert(0, AFML_ROOT)
sys.path.insert(0, os.path.join(AFML_ROOT, 'ch19', 'microstructural_features'))

INPUT_DATA = os.path.join(AFML_ROOT, 'input_data')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import microstructural_features as mf

DOLLAR_BAR_THRESHOLD = 10000.0

## Part A — Rebuilding the real $10,000 dollar bars

Every feature below needs the same foundation: the actual 249 dollar bars
this whole pipeline is built on (Ch02's $10,000 threshold), with every trade
tagged with the bar it belongs to.

Binance's raw trade file also gives us `IsBuyerMaker` — the TRUE aggressor
side of every trade. `IsBuyerMaker=True` means the seller was aggressive
(sold into a resting bid) → sell-initiated → label −1. `IsBuyerMaker=False`
means the buyer was aggressive → buy-initiated → label +1. That's ground
truth, not an inference — we'll use it both directly (as the real signed-volume
input to several features) and as a way to check how accurate the book's
tick-rule *estimate* of trade direction actually is on this data.

In [ ]:
trades_path = os.path.join(INPUT_DATA, 'BTCTUSD-trades-2026-03.csv')
raw = pd.read_csv(trades_path, header=None,
                   names=['TradeID', 'Price', 'Volume', 'QuoteVolume',
                          'Timestamp', 'IsBuyerMaker', 'IsBestMatch'])
raw['Date'] = pd.to_datetime(raw['Timestamp'], unit='us')
raw['Label'] = raw['IsBuyerMaker'].apply(lambda x: -1 if x else 1)
trades = raw[['Date', 'Price', 'Volume', 'Label']].copy()
print(f"Loaded {len(trades)} real trades")

cumm_dollar, bar_id, bar_ids = 0.0, 0, []
for price, volume in zip(trades['Price'], trades['Volume']):
    cumm_dollar += price * volume
    bar_ids.append(bar_id)
    if cumm_dollar >= DOLLAR_BAR_THRESHOLD:
        bar_id += 1
        cumm_dollar = 0.0
trades['bar_id'] = bar_ids

n_complete_bars = trades['bar_id'].max()
trades = trades[trades['bar_id'] < n_complete_bars].copy()
print(f"{n_complete_bars} complete $10,000 dollar bars ({len(trades)} trades)")

bars = trades.groupby('bar_id').agg(
    Open=('Price', 'first'), High=('Price', 'max'), Low=('Price', 'min'), Close=('Price', 'last'),
    Volume=('Volume', 'sum'), n_trades=('Price', 'size'),
)
dollar_per_trade = trades['Price'] * trades['Volume']
bars['DollarVolume'] = dollar_per_trade.groupby(trades['bar_id']).sum()
bars['BuyVolume'] = trades[trades['Label'] == 1].groupby('bar_id')['Volume'].sum().reindex(bars.index).fillna(0)
bars['SellVolume'] = trades[trades['Label'] == -1].groupby('bar_id')['Volume'].sum().reindex(bars.index).fillna(0)
bars.head()

## Part B — Each feature, once, over the whole real series

Genuine headline numbers from real BTC/TUSD data — not synthetic placeholders.

### 19.3.1 — Tick rule vs. true side

**Why:** the tick rule infers a trade's aggressor sign from price alone —
useful for datasets that don't give you the true side directly. We have the
true side here (Binance's `IsBuyerMaker`), so we can score the rule's
accuracy directly instead of just citing the literature.

**Math:** `b_t = +1 if Δp_t>0, −1 if Δp_t<0, b_(t-1) if Δp_t==0`, `b_0=+1`.

In [ ]:
inferred_side = mf.tick_rule(trades['Price'].values)
tick_acc = mf.tick_rule_accuracy(inferred_side, trades['Label'].values)
print(f"Tick rule accuracy vs. true side: {tick_acc:.4f}")

### 19.3.2 — Roll model

**Why:** even with nothing but a price series, the bid-ask spread leaves a
fingerprint in how prices bounce back and forth — that bounce shows up as
negative serial covariance in price changes.

**Math:** `c = sqrt(max(0, -Cov[Δp_t, Δp_(t-1)]))`,
`sigma_u = sqrt(max(0, Var[Δp_t] + 2*Cov[Δp_t, Δp_(t-1)]))`.

**Adaptation note:** applied here to bar closes, not trade-to-trade prices
(Roll's model assumes uniformly-spaced ticks; dollar bars aren't) — kept
because bar closes are the only price series consistent with the rest of
this pipeline.

In [ ]:
roll_res = mf.roll_measure(bars['Close'].values)
print(f"effective half-spread c = ${roll_res['c']:.2f}")
print(f"fundamental noise sigma_u = ${roll_res['sigma_u']:.2f}")

### 19.3.3 — Parkinson high-low volatility

**Math:** `sigma_HL = sqrt( mean( (log(H/L))^2 ) / (4*log(2)) )`.

In [ ]:
park_vol = mf.parkinson_volatility(bars['High'].values, bars['Low'].values)
print(f"sigma_HL = {park_vol:.6f}")

### 19.3.4 — Corwin-Schultz spread + Becker-Parkinson sigma (Snippets 19.1/19.2)

**Why:** sharpens the high-low idea into an actual bid-ask spread estimate,
by comparing a 2-bar high/low range (grows with both volatility and time)
against a 1-bar range (only grows with volatility).

**Book-snippet note:** the printed code uses a pre-2016 pandas API
(`pd.stats.moments.rolling_*`) that's long gone — ported 1:1 in spirit to
modern `.rolling()` calls.

In [ ]:
cs_spread = mf.corwin_schultz(bars['High'], bars['Low'], sl=1)
beta = mf.get_beta(bars['High'], bars['Low'], 1)
gamma = mf.get_gamma(bars['High'], bars['Low'])
bp_sigma = mf.becker_parkinson_sigma(beta, gamma)
frac_zero = float((cs_spread == 0).mean())
print(f"mean spread={cs_spread.mean():.6f}, median={cs_spread.median():.6f}")
print(f"{frac_zero:.1%} of bars clipped to 0 spread (alpha<0, book's own rule p.727)")
print(f"Becker-Parkinson sigma: mean={bp_sigma.mean():.6f}")

### 19.4.1 — Kyle's Lambda

**Why:** the price-impact-per-unit-of-signed-volume — how much a market
maker has to move price per dollar of net order flow, just in case it's an
informed trader. High lambda = illiquid / adversely-selected market.

**Math:** `Δp_t = lambda*(b_t*V_t) + epsilon_t`, fit per bar via OLS on that
bar's own trades.

In [ ]:
signed_vol = trades['Volume'] * trades['Label']
kyle = mf.kyle_lambda_by_bar(trades['Price'].values, signed_vol.values,
                              trades['bar_id'].values, min_trades=5)
print(f"{kyle.notna().sum()} of {len(kyle)} bars had >=5 trades to fit")
print(f"mean={kyle.mean():.1f}, median={kyle.median():.1f}, range=[{kyle.min():.1f}, {kyle.max():.1f}]")
frac_negative = float((kyle < 0).mean())
print(f"{frac_negative:.1%} of estimates are NEGATIVE (Kyle's model requires lambda>0 -- "
      "a real small-sample limitation, not a book bug)")

### 19.4.2 — Amihud's Lambda

**Math:** `|Δlog(close_tau)| = lambda * sum_{t in bar tau}(p_t*V_t) + epsilon_tau`
(no intercept — fit through the origin).

In [ ]:
amihud = mf.amihud_lambda(bars['Close'].values, bars['DollarVolume'].values)
print(f"Amihud's Lambda: {amihud:.4e}")

### 19.5.2 — VPIN

**Why:** the practical, high-frequency version of PIN — instead of fitting
an unobservable maximum-likelihood model, just measure how lopsided buy vs.
sell volume is across a rolling window of bars.

**Math:** `VPIN_tau = sum|VB_i - VS_i| / sum(VB_i + VS_i)` over a rolling
n-bar window.

In [ ]:
vpin10 = mf.vpin(bars['BuyVolume'].values, bars['SellVolume'].values, window=10)
print(f"mean VPIN (10-bar window) = {vpin10.mean():.4f}, latest = {vpin10.iloc[-1]:.4f}")

### 19.6.1 — Round-number order-size frequency (adapted)

**Adaptation:** the book's finding is about discrete equity contract counts.
BTC quantities are continuous, so this checks whether trade volume lands
near a set of plausible "round" levels instead.

In [ ]:
rnf = mf.round_number_frequency(trades['Volume'].values)
top_levels = sorted(rnf['by_level'].items(), key=lambda kv: -kv[1])[:5]
print(f"round_fraction = {rnf['round_fraction']:.4f}")
print(f"top levels: {top_levels}")

### 19.6.5 — Serial correlation of signed order flow

**Why:** positive autocorrelation in {b_t} is consistent with either
informed traders acting on persistent information, or large orders being
split into smaller pieces (the book argues the latter dominates at short
timescales — Toth et al. [2011]).

In [ ]:
sc1 = mf.serial_correlation_signed_flow(trades['Label'].values, lag=1)
sc5 = mf.serial_correlation_signed_flow(trades['Label'].values, lag=5)
print(f"lag-1 autocorr = {sc1:.4f}, lag-5 autocorr = {sc5:.4f}")

### Distributions — printed stats get an accompanying chart (notebook-output convention)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].hist(kyle.dropna(), bins=25, color='steelblue', edgecolor='white')
axes[0, 0].axvline(0, color='black', linewidth=1)
axes[0, 0].set_title("Kyle's Lambda per real bar (Fig 19.1 analogue)")
axes[0, 0].set_xlabel('lambda')

axes[0, 1].hist(cs_spread, bins=25, color='seagreen', edgecolor='white')
axes[0, 1].set_title('Corwin-Schultz spread per real bar')
axes[0, 1].set_xlabel('spread (fraction of price)')

axes[1, 0].plot(vpin10.index, vpin10.values, color='darkorange')
axes[1, 0].set_title('VPIN, 10-bar rolling window')
axes[1, 0].set_xlabel('bar id')

axes[1, 1].plot(trades['bar_id'].values[:2000], signed_vol.cumsum().values[:2000], color='crimson')
axes[1, 1].set_title('Cumulative signed order flow (first 2000 trades)')
axes[1, 1].set_xlabel('bar id')

plt.tight_layout()
plt.savefig('ch19_feature_distributions.png', dpi=110)
plt.show()

## Part C — Full per-bar feature table (Ch07 input, next session)

Builds a 249-row, bar-indexed table with all 9 features (rolling 20-bar
windows for the whole-series estimators, per-bar for the rest) and saves it
to `input_data/ch19_microstructural_features.{csv,pkl}`.

**Not done here:** merging this onto Ch07's 88-event training table (aligning
each triple-barrier event to its bar) — that's the deliberate next step.

In [ ]:
ROLL_WINDOW = 20

def _rolling_bar(a, b, func, window):
    out = [np.nan] * len(a)
    for i in range(window - 1, len(a)):
        try:
            out[i] = func(a[i - window + 1:i + 1], b[i - window + 1:i + 1])
        except Exception:
            out[i] = np.nan
    return out

closes = bars['Close'].values
roll_c, roll_sigma_u = [], []
for i in range(len(closes)):
    if i < ROLL_WINDOW:
        roll_c.append(np.nan); roll_sigma_u.append(np.nan); continue
    res = mf.roll_measure(closes[i - ROLL_WINDOW:i + 1])
    roll_c.append(res['c']); roll_sigma_u.append(res['sigma_u'])

parkinson_roll = _rolling_bar(bars['High'].values, bars['Low'].values, mf.parkinson_volatility, ROLL_WINDOW)
amihud_roll = _rolling_bar(bars['Close'].values, bars['DollarVolume'].values, mf.amihud_lambda, ROLL_WINDOW)

cs_spread_full = mf.corwin_schultz(bars['High'], bars['Low'], sl=1).reindex(bars.index)
bp_sigma_full = mf.becker_parkinson_sigma(
    mf.get_beta(bars['High'], bars['Low'], 1), mf.get_gamma(bars['High'], bars['Low'])
).reindex(bars.index)

kyle_full = mf.kyle_lambda_by_bar(
    trades['Price'].values, signed_vol.values, trades['bar_id'].values, min_trades=5
).reindex(bars.index)

vpin_full = mf.vpin(bars['BuyVolume'].values, bars['SellVolume'].values, window=10)

round_frac, serial_corr, tick_acc_bar = [], [], []
for bid, grp in trades.groupby('bar_id'):
    round_frac.append(mf.round_number_frequency(grp['Volume'].values)['round_fraction'])
    serial_corr.append(
        mf.serial_correlation_signed_flow(grp['Label'].values, lag=1) if len(grp) > 3 else np.nan
    )
    tick_acc_bar.append(
        mf.tick_rule_accuracy(mf.tick_rule(grp['Price'].values), grp['Label'].values)
    )

feature_table = pd.DataFrame({
    'roll_c': roll_c, 'roll_sigma_u': roll_sigma_u,
    'parkinson_vol_20bar': parkinson_roll,
    'corwin_schultz_spread': cs_spread_full.values,
    'becker_parkinson_sigma': bp_sigma_full.values,
    'kyle_lambda': kyle_full.values,
    'amihud_lambda_20bar': amihud_roll,
    'vpin_10bar': vpin_full.values,
    'round_number_fraction': round_frac,
    'serial_corr_signed_flow': serial_corr,
    'tick_rule_accuracy': tick_acc_bar,
}, index=bars.index)
feature_table.index.name = 'bar_id'

print(feature_table.shape)
feature_table.describe().T[['count', 'mean', 'std', 'min', 'max']]

In [ ]:
csv_out = os.path.join(INPUT_DATA, 'ch19_microstructural_features.csv')
pkl_out = os.path.join(INPUT_DATA, 'ch19_microstructural_features.pkl')
feature_table.to_csv(csv_out)
feature_table.to_pickle(pkl_out)
print(f"Saved: {csv_out}")
print(f"Saved: {pkl_out}")

## TDD results

REAL-MACHINE CONFIRMED (Python 3.10.20 / pytest 9.0.3 / mlfinlab env,
2026-07-16) — 41 passed in 0.88s. Real-data headline numbers above are
byte-identical to the sandbox run — nothing here was environment-sensitive.

```
===================================================================== test session starts ======================================================================
test_microstructural_features.py::TestTickRule::test_hand_traced_sequence PASSED                                                                          [  2%]
test_microstructural_features.py::TestTickRule::test_b0_is_configurable PASSED                                                                            [  4%]
test_microstructural_features.py::TestTickRule::test_empty_input PASSED                                                                                   [  7%]
test_microstructural_features.py::TestTickRule::test_single_price PASSED                                                                                  [  9%]
test_microstructural_features.py::TestTickRuleAccuracy::test_hand_traced_5_of_6 PASSED                                                                    [ 12%]
test_microstructural_features.py::TestTickRuleAccuracy::test_perfect_agreement PASSED                                                                     [ 14%]
test_microstructural_features.py::TestTickRuleAccuracy::test_length_mismatch_raises PASSED                                                                [ 17%]
test_microstructural_features.py::TestTickRuleAccuracy::test_empty_raises PASSED                                                                          [ 19%]
test_microstructural_features.py::TestRollMeasure::test_cross_validated_against_numpy PASSED                                                              [ 21%]
test_microstructural_features.py::TestRollMeasure::test_known_regression_values PASSED                                                                    [ 24%]
test_microstructural_features.py::TestRollMeasure::test_too_short_raises PASSED                                                                           [ 26%]
test_microstructural_features.py::TestParkinsonVolatility::test_hand_traced_formula PASSED                                                                [ 29%]
test_microstructural_features.py::TestParkinsonVolatility::test_known_value PASSED                                                                        [ 31%]
test_microstructural_features.py::TestParkinsonVolatility::test_zero_range_gives_zero_vol PASSED                                                          [ 34%]
test_microstructural_features.py::TestParkinsonVolatility::test_length_mismatch_raises PASSED                                                             [ 36%]
test_microstructural_features.py::TestCorwinSchultz::test_cross_validated_against_reference_port PASSED                                                   [ 39%]
test_microstructural_features.py::TestCorwinSchultz::test_negative_alpha_clipped_to_zero_spread PASSED                                                    [ 41%]
test_microstructural_features.py::TestCorwinSchultz::test_known_values PASSED                                                                             [ 43%]
test_microstructural_features.py::TestCorwinSchultz::test_becker_parkinson_sigma_nonnegative PASSED                                                       [ 46%]
test_microstructural_features.py::TestKyleLambda::test_exact_slope_recovered PASSED                                                                       [ 48%]
test_microstructural_features.py::TestKyleLambda::test_cross_validated_against_polyfit PASSED                                                             [ 51%]
test_microstructural_features.py::TestKyleLambda::test_no_variation_in_volume_returns_nan PASSED                                                          [ 53%]
test_microstructural_features.py::TestKyleLambda::test_too_few_points_returns_nan PASSED                                                                  [ 56%]
test_microstructural_features.py::TestKyleLambda::test_length_mismatch_raises PASSED                                                                      [ 58%]
test_microstructural_features.py::TestKyleLambdaByBar::test_known_two_bar_case PASSED                                                                     [ 60%]
test_microstructural_features.py::TestKyleLambdaByBar::test_bar_below_min_trades_is_nan PASSED                                                            [ 63%]
test_microstructural_features.py::TestAmihudLambda::test_cross_validated_closed_form_ols PASSED                                                           [ 65%]
test_microstructural_features.py::TestAmihudLambda::test_known_value PASSED                                                                               [ 68%]
test_microstructural_features.py::TestAmihudLambda::test_zero_volume_returns_nan PASSED                                                                   [ 70%]
test_microstructural_features.py::TestAmihudLambda::test_length_mismatch_raises PASSED                                                                    [ 73%]
test_microstructural_features.py::TestVpin::test_hand_traced_rolling_window PASSED                                                                        [ 75%]
test_microstructural_features.py::TestVpin::test_window_one_gives_per_bar_imbalance_fraction PASSED                                                       [ 78%]
test_microstructural_features.py::TestVpin::test_length_mismatch_raises PASSED                                                                            [ 80%]
test_microstructural_features.py::TestVpin::test_zero_window_raises PASSED                                                                                [ 82%]
test_microstructural_features.py::TestRoundNumberFrequency::test_hand_traced_matches PASSED                                                               [ 85%]
test_microstructural_features.py::TestRoundNumberFrequency::test_custom_levels PASSED                                                                     [ 87%]
test_microstructural_features.py::TestRoundNumberFrequency::test_empty_raises PASSED                                                                      [ 90%]
test_microstructural_features.py::TestSerialCorrelationSignedFlow::test_perfect_alternation_gives_minus_one PASSED                                        [ 92%]
test_microstructural_features.py::TestSerialCorrelationSignedFlow::test_cross_validated_against_pandas_autocorr PASSED                                    [ 95%]
test_microstructural_features.py::TestSerialCorrelationSignedFlow::test_lag_2 PASSED                                                                      [ 97%]
test_microstructural_features.py::TestSerialCorrelationSignedFlow::test_too_short_raises PASSED                                                           [100%]

====================================================================== 41 passed in 0.88s ======================================================================
```